# INITIAL COMMIT

Imports

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

Reading the data

In [3]:
df = pd.read_csv("data/UNSW_NB15_training-set.csv")

### Feature engineering:

#### Dropping certain features
- "id" is an index and does not help with categorization
- "attack_cat" is an extension of a label and should not be included in the training data
- "stcpb" and "dtcpb" source and destination TCP base sequence number is randomly initialized and does not help with categorization

In [4]:
df.drop(["id", "attack_cat", "stcpb", "dtcpb"], axis=1, inplace=True)

#### One Hot Encoding columns with non-numerical data:
1. "proto" - the column denoting the protocol of the packet
2. "state" - state of the packet
3. "service" - 

In [5]:
# create map of groups to protocol
protocol_groups = {
    'common_transport': ['tcp', 'udp', 'sctp'],
    'routing': ['ospf', 'eigrp', 'egp', 'igp', 'nsfnet-igp', 'dgp', 'idrp', 'idpr', 'idpr-cmtp', 'sdrp', 'mhrp'],
    'tunneling': ['gre', 'ipip', 'l2tp', 'encap', 'etherip', 'mobile', 'ipcomp', 'ipnip', 'ip', 'micp'],
    'ipv6_family': ['ipv6', 'ipv6-frag', 'ipv6-route', 'ipv6-no', 'ipv6-opts'],
    'multicast': ['igmp', 'pim', 'vrrp', 'pgm', 'cbt'],
    'link_layer': ['arp', 'ax.25', 'fc', 'srp', 'il', 'ipx-n-ip'],
    'security': ['skip', 'tlsp', 'rsvp', 'kryptolan', 'secure-vmtp', 'aes-sp3-d', 'swipe', 'pri-enc'],
    'legacy': ['ggp', 'st2', 'argus', 'chaos', 'nvp', 'pup', 'xnet', 'mux', 'dcn', 'hmp', 'prm', 'trunk-1', 'trunk-2', 'xns-idp', 'leaf-1', 'leaf-2', 'irtp', 'rdp', 'netblt', 'mfe-nsp', 'merit-inp', '3pc', 'ddp', 'tp++', 'narp', 'any', 'cftp', 'sat-expak', 'ippc', 'sat-mon', 'cpnx', 'wsn', 'pvp', 'br-sat-mon', 'sun-nd', 'wb-mon', 'vmtp', 'ttp', 'vines', 'tcf', 'sprite-rpc', 'larp', 'mtp', 'bbn-rcc', 'bna', 'visa', 'ipcv', 'cphb', 'iso-tp4', 'wb-expak', 'sep', 'xtp', 'unas', 'iso-ip', 'aris', 'a/n', 'snp', 'compaq-peer', 'zero', 'ddx', 'iatp', 'stp', 'uti', 'sm', 'smp', 'isis', 'ptp', 'fire', 'crtp', 'crudp', 'sccopmce', 'iplt', 'pipe', 'sps', 'ib', 'emcon', 'gmtp', 'ifmp', 'pnni', 'qnx', 'scps']
}

# create a mapping of protocols to group
proto_to_group = {proto: group for group, protos in protocol_groups.items() for proto in protos}

# map protocol to their respective group
df['proto_group'] = df['proto'].map(proto_to_group).fillna('legacy')


In [6]:
# one hot encode protocol group
df_ohe = pd.get_dummies(df, columns=['proto_group', 'state', 'service'])

# convert boolean columns to int
bool_cols = df_ohe.select_dtypes(include='bool').columns
df_ohe[bool_cols] = df_ohe[bool_cols].astype(int)
df_ohe.drop(columns=["proto"], inplace=True)

#### Training/validation/test split

We split before normalizing so that we are able to calculate the averages of only the training set, and apply the transformations to the test and validation sets to prevent information leakage
- `stratify` used to ensure that the training/validation/test datasets will have an equal proportion of class labels as the original dataset. This is especially important for anormaly detection where there is a heavy class imbalance

In [7]:
# Remove labels before processing data
y = df_ohe.pop("label").values.astype(np.float32)
X = df_ohe

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 65865 | Val: 8233 | Test: 8234


#### Normalizing numerical columns

Inspecting the data
- Commented out describe to reduce file size

In [9]:
# Display all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# df_numeric = df_ohe.select_dtypes(include=[np.number])
# df_numeric.describe(include='all')

Different columns need to be handled differently based on what information they represent and the skewedness of their data.
- `Log + StandardScalar` - for columns that are heavily skewed to the right
- `MixMaxScalar` - for columns that we know the hard boundaries for
- `Clip + Log + StandardScalar` - for columns that have extreme values that are deemed to be noise and contribute in no meaningful way to training
- `StandardScalar` - for columns that already have a roughly normal distribution

In [10]:
log_scale_cols = ['dur', 'sbytes', 'dbytes', 'spkts', 'dpkts', 'sload', 'dload', 'rate', 'dttl', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'smean', 'dmean', 'ct_flw_http_mthd', 'trans_depth', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm']

std_scale_cols = ['tcprtt', 'synack', 'ackdat']

minmax_cols = ['sttl', 'swin', 'dwin']

# Columns needing clip before log
clip_cols = {
    'sloss': 0.99,
    'dloss': 0.99,
    'response_body_len': 0.99
}

# Apply clipping before the pipeline
for col, quantile in clip_cols.items():
    cap = X_train[col].quantile(quantile)  # fit on train only
    X_train[col] = X_train[col].clip(upper=cap)
    X_val[col]   = X_val[col].clip(upper=cap)
    X_test[col]  = X_test[col].clip(upper=cap)

# Add clipped columns to log group
log_scale_cols += list(clip_cols.keys())

# Add binary flag for response_body_len before dropping into pipeline
df_ohe['has_response_body'] = (df_ohe['response_body_len'] > 0).astype(np.float32)

# Build pipeline for Log + StandardScalar
log_transformer = Pipeline([
    ('log1p',  FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('log_scale',   log_transformer,  log_scale_cols),
    ('std_scale',   StandardScaler(), std_scale_cols),
    ('minmax',      MinMaxScaler(),   minmax_cols)
], remainder='passthrough')

# Fit on train only, transform all splits
X_train = preprocessor.fit_transform(X_train).astype(np.float32)
X_val = preprocessor.transform(X_val).astype(np.float32)
X_test = preprocessor.transform(X_test).astype(np.float32)

### Dataset and DataLoader

Dataset Class

In [11]:
class PacketsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

DataLoaders

In [12]:
train_dataset = PacketsDataset(X_train, y_train)
val_dataset = PacketsDataset(X_val,   y_val)
test_dataset = PacketsDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)